# ScatterBrain -- Form Factor Fitting Tutorial

This notebook demonstrates fitting form factors to synthetic 1D SAXS data using the
Phase 2 features:

1. Generate synthetic data for a cylinder and a core-shell sphere
2. Fit a cylinder form factor (`cylinder_pq`) with `fit_model`
3. Fit a core-shell sphere form factor (`core_shell_sphere_pq`) with `fit_model`
4. Inspect lmfit confidence intervals from the result dictionary
5. Compute the scattering invariant Q* for the fitted curve

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')  # use non-interactive backend; remove for interactive sessions
import matplotlib.pyplot as plt

import scatterbrain
from scatterbrain.core import ScatteringCurve1D
from scatterbrain.modeling.form_factors import cylinder_pq, core_shell_sphere_pq
from scatterbrain.modeling.fitting import fit_model
from scatterbrain.analysis import guinier_fit, porod_analysis, scattering_invariant
from scatterbrain.visualization import plot_iq, plot_fit, plot_kratky

scatterbrain.configure_logging()
print(f"ScatterBrain version: {scatterbrain.__version__}")

## 1. Cylinder form factor fit

We generate synthetic data for a cylinder with:
- radius R = 4 nm
- length L = 20 nm
- scale = 500, background = 2

Then we fit `fit_model` using `cylinder_pq` to recover these parameters.

In [ ]:
# True parameters
TRUE_CYL_RADIUS = 4.0   # nm
TRUE_CYL_LENGTH = 20.0  # nm
TRUE_SCALE = 500.0
TRUE_BG = 2.0

rng = np.random.default_rng(42)
q_cyl = np.geomspace(0.01, 1.0, 120)

i_cyl_true = TRUE_SCALE * cylinder_pq(q_cyl, TRUE_CYL_RADIUS, TRUE_CYL_LENGTH) + TRUE_BG
noise_frac = 0.03  # 3% relative noise
err_cyl = noise_frac * i_cyl_true
i_cyl_obs = i_cyl_true + rng.normal(0.0, err_cyl)

curve_cyl = ScatteringCurve1D(
    q=q_cyl,
    intensity=i_cyl_obs,
    error=err_cyl,
    q_unit="nm^-1",
    intensity_unit="a.u.",
    metadata={"source": "synthetic_cylinder"},
)

fig, ax = plot_iq(curve_cyl, title="Synthetic cylinder data")
plt.savefig("cyl_raw.png", dpi=100)
plt.show()
print(f"Data points: {len(q_cyl)}, q range: {q_cyl.min():.3f} -- {q_cyl.max():.3f} nm^-1")

In [ ]:
# Fit cylinder form factor
# initial_params order: [scale, background, radius, length]
cyl_fit = fit_model(
    curve_cyl,
    model_func=cylinder_pq,
    param_names=["radius", "length"],
    initial_params=[400.0, 1.0, 5.0, 15.0],
    param_bounds=([0, 0, 0.1, 0.1], [1e5, 100, 50, 200]),
)

assert cyl_fit is not None, "Fit returned None -- unexpected failure"

fp = cyl_fit["fitted_params"]
se = cyl_fit["fitted_params_stderr"]
print(f"Fit success : {cyl_fit['success']}")
print(f"chi^2_red   : {cyl_fit['chi_squared_reduced']:.3f}")
print(f"scale       : {fp['scale']:.2f} +/- {se['scale']:.2f}  (true: {TRUE_SCALE})")
print(f"background  : {fp['background']:.2f} +/- {se['background']:.2f}  (true: {TRUE_BG})")
print(f"radius      : {fp['radius']:.3f} +/- {se['radius']:.3f} nm  (true: {TRUE_CYL_RADIUS})")
print(f"length      : {fp['length']:.3f} +/- {se['length']:.3f} nm  (true: {TRUE_CYL_LENGTH})")

In [ ]:
# Plot fit
fig = plot_fit(curve_cyl, cyl_fit, plot_residuals=True, title="Cylinder Form Factor Fit")
plt.savefig("cyl_fit.png", dpi=100)
plt.show()

In [ ]:
# Inspect lmfit confidence intervals (1-sigma)
ci = cyl_fit.get("confidence_intervals")
if ci is not None:
    print("1-sigma confidence intervals:")
    for pname, (lo, hi) in ci.items():
        print(f"  {pname:12s}: [{lo:.4g}, {hi:.4g}]")
else:
    print("Confidence intervals not available (lmfit not installed or fit did not converge).")

## 2. Core-shell sphere form factor fit

We generate synthetic data for a core-shell sphere with:
- core radius R_c = 5 nm, shell thickness t = 2 nm
- contrast_core = 1.0, contrast_shell = 0.3
- scale = 800, background = 1

In [ ]:
# True parameters
TRUE_RC = 5.0    # nm core radius
TRUE_T  = 2.0    # nm shell thickness
TRUE_RHO_C = 1.0
TRUE_RHO_S = 0.3
TRUE_CS_SCALE = 800.0
TRUE_CS_BG = 1.0

q_cs = np.geomspace(0.01, 0.8, 100)
i_cs_true = TRUE_CS_SCALE * core_shell_sphere_pq(
    q_cs, TRUE_RC, TRUE_T, TRUE_RHO_C, TRUE_RHO_S
) + TRUE_CS_BG
err_cs = 0.03 * i_cs_true
i_cs_obs = i_cs_true + rng.normal(0.0, err_cs)

curve_cs = ScatteringCurve1D(
    q=q_cs,
    intensity=i_cs_obs,
    error=err_cs,
    q_unit="nm^-1",
    intensity_unit="a.u.",
    metadata={"source": "synthetic_core_shell"},
)

fig, ax = plot_iq(curve_cs, title="Synthetic core-shell sphere data")
plt.savefig("cs_raw.png", dpi=100)
plt.show()

In [ ]:
# Fix contrasts at true values; fit only scale, background, radius_core, shell_thickness
def cs_pq_fixed_contrast(q, radius_core, shell_thickness):
    return core_shell_sphere_pq(q, radius_core, shell_thickness,
                                contrast_core=TRUE_RHO_C,
                                contrast_shell=TRUE_RHO_S)

cs_fit = fit_model(
    curve_cs,
    model_func=cs_pq_fixed_contrast,
    param_names=["radius_core", "shell_thickness"],
    initial_params=[600.0, 0.5, 4.0, 1.5],
    param_bounds=([0, 0, 0.5, 0], [1e4, 50, 20, 10]),
)

assert cs_fit is not None, "Fit returned None -- unexpected failure"

fp = cs_fit["fitted_params"]
se = cs_fit["fitted_params_stderr"]
print(f"Fit success    : {cs_fit['success']}")
print(f"chi^2_red      : {cs_fit['chi_squared_reduced']:.3f}")
print(f"scale          : {fp['scale']:.2f} +/- {se['scale']:.2f}  (true: {TRUE_CS_SCALE})")
print(f"background     : {fp['background']:.3f} +/- {se['background']:.3f}  (true: {TRUE_CS_BG})")
print(f"radius_core    : {fp['radius_core']:.3f} +/- {se['radius_core']:.3f} nm  (true: {TRUE_RC})")
print(f"shell_thickness: {fp['shell_thickness']:.3f} +/- {se['shell_thickness']:.3f} nm  (true: {TRUE_T})")

In [ ]:
fig = plot_fit(curve_cs, cs_fit, plot_residuals=True, title="Core-Shell Sphere Form Factor Fit")
plt.savefig("cs_fit.png", dpi=100)
plt.show()

In [ ]:
ci_cs = cs_fit.get("confidence_intervals")
if ci_cs is not None:
    print("1-sigma confidence intervals (core-shell fit):")
    for pname, (lo, hi) in ci_cs.items():
        print(f"  {pname:20s}: [{lo:.4g}, {hi:.4g}]")
else:
    print("Confidence intervals not available.")

## 3. Kratky plot

The Kratky plot ($I(q) q^2$ vs $q$) is useful for assessing particle compactness.
A globular (compact) particle shows a bell-shaped peak; an unfolded / rod-like
particle shows a plateau or monotonically increasing curve.

In [ ]:
# Guinier fit to the cylinder data for the normalized Kratky plot
g_cyl = guinier_fit(curve_cyl, q_range=(0.01, 0.15))
if g_cyl is not None:
    print(f"Guinier Rg (cylinder) = {g_cyl['Rg']:.3f} nm")

fig, ax = plot_kratky(curve_cyl, guinier_result=g_cyl, normalized=True,
                      title="Dimensionless Kratky Plot -- Cylinder")
plt.savefig("kratky_cyl.png", dpi=100)
plt.show()

## 4. Scattering invariant Q*

The scattering invariant
$$Q^* = \int_0^\infty q^2 I(q)\,dq$$
is computed numerically with Guinier low-q and Porod high-q extrapolations.

In [ ]:
# Use the fitted curve (noiseless model) to compute Q*
fit_curve_cyl = cyl_fit["fit_curve"]

g_fit = guinier_fit(fit_curve_cyl, q_range=(0.01, 0.15))
p_fit = porod_analysis(fit_curve_cyl, q_fraction_high=0.2)

inv_result = scattering_invariant(
    fit_curve_cyl,
    guinier_result=g_fit,
    porod_result=p_fit,
)

if inv_result is not None:
    print(f"Q* (data region) : {inv_result['Q_star']:.4g}")
    print(f"Q* (low-q extra) : {inv_result['Q_star_low_q']:.4g}")
    print(f"Q* (high-q extra): {inv_result['Q_star_high_q']:.4g}")
    print(f"Q* (total)       : {inv_result['Q_star_total']:.4g}")
else:
    print("Scattering invariant could not be computed.")

## Summary

| Parameter | True | Fitted (cylinder) |
|-----------|------|-------------------|
| scale | 500 | see above |
| background | 2 | see above |
| radius (nm) | 4 | see above |
| length (nm) | 20 | see above |

| Parameter | True | Fitted (core-shell) |
|-----------|------|---------------------|
| scale | 800 | see above |
| background | 1 | see above |
| radius_core (nm) | 5 | see above |
| shell_thickness (nm) | 2 | see above |